# Notebook 3 . Gabungkan Baris dari Banyak CSV

Notebook ini menumpuk isi beberapa CSV ke bawah (append), misalnya file penjualan per bulan. Setiap file idealnya memiliki nama kolom yang sama. Baris sumber tidak diubah.

In [ ]:
from pathlib import Path
import pandas as pd

# PENGATURAN YANG BISA DIUBAH
Daftar_Input_CSV = [
    "Input/januari.csv",
    "Input/februari.csv",
    # "Input/maret.csv",
]
Separator_CSV = ","  # Ganti menjadi ";" jika CSV memakai titik koma
Samakan_Kolom = True  # Satukan kolom yang berbeda; sel kosong diisi untuk kolom yang tidak ada
Output_CSV = Path("Output/hasil_append.csv")

def find_project_dir():
    current = Path.cwd().resolve()
    for folder in (current, *current.parents):
        if (folder / "Input").is_dir():
            (folder / "Output").mkdir(parents=True, exist_ok=True)
            return folder
    return current

BASE_DIR = find_project_dir()


## Gabungkan dan simpan

In [ ]:
frames = []
for filename in Daftar_Input_CSV:
    path = Path(filename)
    if not path.is_absolute(): path = BASE_DIR / path
    if not path.exists(): raise FileNotFoundError(f"File tidak ditemukan: {path}")
    frame = pd.read_csv(path, sep=Separator_CSV, dtype="string", encoding="utf-8-sig")
    frame.columns = frame.columns.astype(str).str.strip()
    frame["_sumber_file"] = path.name
    print(f"{path.name}: {len(frame):,} baris, {len(frame.columns)-1} kolom sumber")
    frames.append(frame)
if not frames: raise ValueError("Tambahkan minimal satu file ke Daftar_Input_CSV.")
if Samakan_Kolom:
    result = pd.concat(frames, ignore_index=True, sort=False)
else:
    reference = list(frames[0].columns)
    for i, frame in enumerate(frames[1:], start=2):
        if list(frame.columns) != reference:
            raise ValueError(f"Susunan kolom file ke-{i} berbeda. Aktifkan Samakan_Kolom atau samakan kolomnya.")
    result = pd.concat(frames, ignore_index=True)
output = Output_CSV if Output_CSV.is_absolute() else BASE_DIR / Output_CSV
output.parent.mkdir(parents=True, exist_ok=True)
result.to_csv(output, index=False, encoding="utf-8-sig", sep=Separator_CSV)
print(f"Hasil: {len(result):,} baris; tersimpan di {output.resolve()}")
display(result.head())
